In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# -----------------------------
# Residual Block with LayerNorm
# -----------------------------
class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        # self.bn = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        identity = x
        out = self.relu(self.fc1(x))
        out = self.dropout(out)
        out = self.relu(self.fc2(out))
        return out + identity

# -----------------------------
# Full DNN Model
# -----------------------------
class SpeciesDNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_species, dropout):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.ln_input = nn.LayerNorm(hidden_dim)

        self.residual_blocks = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
        )

        self.output_layer = nn.Linear(hidden_dim, n_species)

    def forward(self, x):
        x = self.ln_input(self.input_layer(x))
        x = self.residual_blocks(x)
        x = self.output_layer(x)
        return x



In [ ]:
dir_path = Path.cwd().resolve().parents[0]
data_folder = dir_path.joinpath("data/to_model")
model_path = dir_path.joinpath('models','species_dnn.pt')
pred_csv_path = data_folder.joinpath("predictions.csv")

# Load data
X_df = pd.read_parquet(data_folder.joinpath("X.parquet"))
y_df = pd.read_parquet(data_folder.joinpath("Y.parquet"))

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X = torch.tensor(X_df.values, dtype=torch.float32).to(device)
y = torch.tensor(y_df.values, dtype=torch.int64).to(device)
y = y.view(-1).long()

In [ ]:
input_dim = X_df.shape[1]
n_species = 14
hidden_dim = 64          # <<< FIXED SIZE
batch_size = 128
epochs = 300
learning_rate = 0.001    # <<< LOWER LR
dropout = 0.2


In [ ]:
# Model
model = SpeciesDNN(input_dim, hidden_dim, n_species, dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# DataLoader
dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

best_loss = float("inf")
patience = 5
counter = 0


In [ ]:
# Training loop
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()

        # Gradient clipping (extra safety)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        epoch_loss += loss.item() * X_batch.size(0)

    epoch_loss /= len(loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.6f}")

    # ---- Early Stopping Logic ----
    if epoch_loss < best_loss:
        best_loss = epoch_loss
    #     counter = 0
        torch.save(model.state_dict(), model_path)
    # else:
    #     counter += 1

    # if counter >= patience:
    #     break


## TESTING

In [ ]:
# Load data
X_test = pd.read_parquet(data_folder.joinpath("X_test.parquet"))
y_test = pd.read_parquet(data_folder.joinpath("y_test.parquet"))

X = torch.tensor(X_test.values, dtype=torch.float32)
y = torch.tensor(y_test.values, dtype=torch.float32)

y = y.view(-1).long()

dataset = TensorDataset(X, y)
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)


In [ ]:
# Recreate the model architecture FIRST
model = SpeciesDNN(input_dim, hidden_dim, n_species, dropout)

# Load weights
model.load_state_dict(torch.load(model_path, map_location=device))

# Move to device
model.to(device)

model.eval()

# ------------------ Inference ------------------
all_preds = []
all_true = []

with torch.no_grad():
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        logits = model(X_batch)

        # Predictions (multi-class)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.append(preds.cpu())
        all_true.append(y_batch.cpu())

y_pred = torch.cat(all_preds)
y_true = torch.cat(all_true)



In [ ]:

# Accuracy
accuracy = (y_pred == y_true).float().mean().item()
print(f"\nOverall Accuracy: {accuracy:.4f}")

In [ ]:

# Confusion Matrix
conf_mat = torch.zeros(n_species, n_species, dtype=torch.int64)
for t, p in zip(y_true.cpu(), y_pred.cpu()):
    conf_mat[t.item(), p.item()] += 1

print("\nConfusion Matrix (rows=true, cols=pred):")
print(conf_mat)

# Per-class accuracy
per_class_acc = conf_mat.diag() / conf_mat.sum(dim=1).clamp(min=1)

print("\nPer-species accuracy:")
for i, acc in enumerate(per_class_acc):
    print(f"Species {i}: {acc.item():.4f}")

# Save predictions
pred_df = pd.DataFrame({
    "true_species": y_true.cpu().numpy(),
    "pred_species": y_pred.cpu().numpy()
})

pred_df.to_csv(pred_csv_path, index=False)



In [ ]:
sns.heatmap(conf_mat.numpy(), annot=False, cmap="viridis")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap")
plt.show()

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef

print(classification_report(y_true, y_pred))  # Precision, Recall, F1

print("Balanced Accuracy:", balanced_accuracy_score(y_true, y_pred))
print("Cohen Kappa:", cohen_kappa_score(y_true, y_pred))
print("MCC:", matthews_corrcoef(y_true, y_pred))
